# Pipeline SDC — exécution interactive

Notebook pour lancer le pipeline `metadata-analysis-llm-for-sdc` (branche `2-modifications-jawad`)
sans repasser par la ligne de commande, et pour évaluer la qualité d'un output contre un CSV de référence.

**Prérequis** : variable d'environnement `OPENAI_API_KEY` ou `CLE_API_OPENWEBUI` définie (clé LLM SSP Cloud).
Si vos fichiers d'entrée/référence sont sur MinIO (`s3://...`), il faut aussi `AWS_ACCESS_KEY_ID`,
`AWS_SECRET_ACCESS_KEY`, `AWS_SESSION_TOKEN`.


In [1]:
import sys
from pathlib import Path

# Détecte la racine du repo (fonctionne que le notebook soit à la racine ou dans un sous-dossier)
for _p in [Path.cwd(), Path.cwd().parent]:
    if (_p / "src").is_dir():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Impossible de localiser 'src/'. Placez ce notebook a la racine du repo (ou un niveau en dessous).")

sys.path.insert(0, str(REPO_ROOT))

from src.data import read_file
from src.clean import _dataframe_to_rows, clean_sheet
from src.transform_input import wrap, to_markdown
from src.LLM_API_call import chat, is_auto_continued
from src.extract_JSON_array import extract_array
from src.validate_json_output import validate
from src.transform_output import _spanning_pairs, max_spanning, HEADER_BASE

print(f"REPO_ROOT = {REPO_ROOT}")


REPO_ROOT = /home/onyxia/work/metadata-analysis-llm-for-sdc


## Configuration

Renseignez le fichier d'entree, le chemin de sortie, et (optionnel) un CSV de reference
pour l'evaluation de qualite en fin de notebook.


In [2]:
INPUT_PATH  = "s3://jawadmallat/analyse_LLM_metadata/data_tables/metadonnees_ex5_sansSL2.ods"     # .ods / .xlsx / .csv, local ou s3://...
OUTPUT_PATH = "s3://jawadmallat/analyse_LLM_metadata/6juillet/metadonnees_ex5_sansSL2.csv"          # sortie du pipeline
PROMPT_PATH = "src/prompts/prompt_questions.md"

# Optionnel : CSV de reference (sortie attendue, corrigee a la main) pour l'evaluation de qualite.
# Laisser a None pour sauter cette etape.
REFERENCE_CSV = "s3://jawadmallat/analyse_LLM_metadata/data_tables/correction/metadonnees_ex5_sansSL_correction.csv"   # ex: "path/to/correction.csv"


## I-III. Lecture, nettoyage, mise en Markdown


In [3]:
from IPython.display import Markdown, display

# ---- I. Read ---------------------------------------------------
data = read_file(INPUT_PATH)

# ---- II. Clean ---------------------------------------------------
cleaned_sheets = []
for name, sheet_df in data.items():
    rows = _dataframe_to_rows(sheet_df)
    rows = clean_sheet(rows)
    if any(any(c for c in r) for r in rows):
        cleaned_sheets.append((name, rows))

# ---- III. Transform to markdown ------------------------------------
title = INPUT_PATH.rsplit("/", 1)[-1]
markdown_input = to_markdown(cleaned_sheets, title=title)

print(f"{len(markdown_input)} caracteres -- apercu :")
display(Markdown(markdown_input))


28372 caracteres -- apercu :


# metadonnees_ex5_sansSL2.ods


## Tableaux

| Nom du tableau | Libellé | Note Haut | Note Bas | Variables en ligne | Variables en colonne | Variables en page | Décimales |
| --- | --- | --- | --- | --- | --- | --- | --- |
| naf_T1 | Consommation d'énergie en milliers de tonnes-équivalent-pétrole (kTEP) et nombre d'établissements selon le secteur d'activité en NAF. rév.2 |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T1 |  | 1 |
| naf_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyens des produits énergétiques selon le secteur d'activité en NAF rév.2 de l'établissement |  | Note : la consommation d'énergie peut différer de la quantité achetée dans le cas d'énergies stockables (fioul par exemple), ou dans le cas de l'électricité (autoproduction consommée) et du bois (une partie non achetée).Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T2 |  | 1 |
| naf_T3 | Répartition de la consommation de combustibles par usage en milliers de tonnes-équivalent-pétrole (kTEP) selon le secteur d'activité en NAF rév.2 de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T3 |  | 1 |
| naf_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh selon le secteur d'activité en NAF rév.2 de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | SECTEUR | INDIC_T4 |  | 1 |
| reg_T1 | Consommation d'énergie en milliers de tonnes-équivalent-pétrole (kTEP) et nombre d'établissements selon la région |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T1 |  | 1 |
| reg_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyens des produits énergétiques selon la région |  | Note : la consommation d'énergie peut différer de la quantité achetée dans le cas d'énergies stockables (fioul par exemple), ou dans le cas de l'électricité (autoproduction consommée) et du bois (une partie non achetée).Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T2 |  | 1 |
| reg_T3 | Répartition de la consommation de combustibles par usage en milliers de tonnes-équivalent-pétrole (kTEP) selon la région |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T3 |  | 1 |
| reg_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh selon la région |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | REGION | INDIC_T4 |  | 1 |
| teff_T1 | Consommation d'énergie en milliers de tonnes-équivalent-pétrole (kTEP) et nombre d'établissements selon la tranche d'effectif de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T1 |  | 1 |
| teff_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyens des produits énergétiques selon la tranche d'effectif de l'établissement |  | Note : la consommation d'énergie peut différer de la quantité achetée dans le cas d'énergies stockables (fioul par exemple), ou dans le cas de l'électricité (autoproduction consommée) et du bois (une partie non achetée).Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T2 |  | 1 |
| teff_T3 | Répartition de la consommation de combustibles par usage en milliers de tonnes-équivalent-pétrole (kTEP) selon la tranche d'effectif de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T3 |  | 1 |
| teff_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh selon la tranche d'effectif de l'établissement |  | Champ : France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération.Source : Insee, Enquête annuelle sur les consommations d'énergie dans l'industrie 2021. | TEFF | INDIC_T4 |  | 1 |

## Variables

| Variable | Description | Modalité | Définition | Regroupement | Libellé 1 |
| --- | --- | --- | --- | --- | --- |
| REGION | Région | global |  | 11 24 27 28 32 44 52 53 75 76 84 93 97 | Toutes régions |
| REGION | Région | 11 |  |  | Île-de-France |
| REGION | Région | 24 |  |  | Centre-Val de Loire |
| REGION | Région | 27 |  |  | Bourgogne-Franche-Comté |
| REGION | Région | 28 |  |  | Normandie |
| REGION | Région | 32 |  |  | Hauts-de-France |
| REGION | Région | 44 |  |  | Grand Est |
| REGION | Région | 52 |  |  | Pays de la Loire |
| REGION | Région | 53 |  |  | Bretagne |
| REGION | Région | 75 |  |  | Nouvelle-Aquitaine |
| REGION | Région | 76 |  |  | Occitanie |
| REGION | Région | 84 |  |  | Auvergne-Rhône-Alpes |
| REGION | Région | 93 |  |  | Provence-Alpes-Côte d'Azur et Corse |
| REGION | Région | 97 |  |  | Dom |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | global |  | 07 08 09 10 11 12 13 14 15 16 17 18 20 21 22 23 24 25 26 27 28 29 30 31 32 33 38 | Total industrie |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 07-09 |  |  | 07 - 08 – 09 Toutes les Industries extractives |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 10-12 |  |  | 10 – 11 – 12 Industries alimentaires, Fabrication de boissons, Fabrication de produits à base de tabac |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 13 |  |  | 13 - Fabrication de textiles |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 14 |  |  | 14 - Industrie de l'habillement |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 15 |  |  | 15 - Industrie du cuir et de la chaussure |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 16 |  |  | 16 - Travail du bois et fabrication d'articles en bois et en liège, à l'exception des meubles - fabrication d'articles en vannerie et sparterie |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 17 |  |  | 17 - Industrie du papier et du carton |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 18 |  |  | 18 - Imprimerie et reproduction d'enregistrements |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 20 |  |  | 20 - Industrie chimique |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 21 |  |  | 21 - Industrie pharmaceutique |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 22 |  |  | 22 - Fabrication de produits en caoutchouc et en plastique |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 23 |  |  | 23 - Fabrication d'autres produits minéraux non métalliques |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 24 |  |  | 24 - Métallurgie |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 25 |  |  | 25 - Fabrication de produits métalliques, à l'exception des machines et des équipements |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 26 |  |  | 26 - Fabrication de produits informatiques, électroniques et optiques |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 27 |  |  | 27 - Fabrication d'équipements électriques |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 28 |  |  | 28 - Fabrication de machines et équipements n.c.a. |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 29 |  |  | 29 - Industrie automobile |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 30 |  |  | 30 - Fabrication d'autres matériels de transport |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 31 |  |  | 31 - Fabrication de meubles |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 32 |  |  | 32 - Autres industries manufacturières |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 33 |  |  | 33 - Réparation et installation de machines et d'équipements |
| SECTEUR | Secteur d'activité, division de la NAF rév.2 | 38 |  |  | 38 - Récupération |
| TEFF | Tranche d'effectif salarié | global | Intitulé | 03 04 05 06 07 | Total industrie |
| TEFF | Tranche d'effectif salarié | 03 |  |  | 20 à 49 salariés |
| TEFF | Tranche d'effectif salarié | 04 |  |  | 50 à 99 salariés |
| TEFF | Tranche d'effectif salarié | 05 |  |  | 100 à 249 salariés |
| TEFF | Tranche d'effectif salarié | 06 |  |  | 250 à 499 salariés |
| TEFF | Tranche d'effectif salarié | 07 |  |  | 500 salariés ou plus |

## Indicateurs

| Variable | Libellé | Modalité | Notes des modalités | Décimales | Libellé 1 |
| --- | --- | --- | --- | --- | --- |
| INDIC_T1 | Consommation d'énergie en ktep | NB_ETAB |  | 0 | Nombre d'établissements |
| INDIC_T1 | Consommation d'énergie en ktep | CMSR_CSTP | Total des combustibles minéraux solides = houille + lignite charbon pauvre + coke de houille | 1 | Combustibles minéraux solides (houille, lignite - charbon pauvre, coke de houille) |
| INDIC_T1 | Consommation d'énergie en ktep | GAZR_CSTP |  | 1 | Gaz naturel et autres gaz |
| INDIC_T1 | Consommation d'énergie en ktep | BIOR_CSTP |  | 1 | Biogaz et biométhane |
| INDIC_T1 | Consommation d'énergie en ktep | IR_CSTP |  | 1 | Coke de pétrole |
| INDIC_T1 | Consommation d'énergie en ktep | JR_CSTP |  | 1 | Butane propane |
| INDIC_T1 | Consommation d'énergie en ktep | KR_CSTP |  | 1 | Fioul lourd |
| INDIC_T1 | Consommation d'énergie en ktep | LR_CSTP |  | 1 | Fioul domestique |
| INDIC_T1 | Consommation d'énergie en ktep | QR_CSTP |  | 1 | Gazole non routier |
| INDIC_T1 | Consommation d'énergie en ktep | CUR_CSTP | Total combustibles usuels = Combustibles minéraux solides + gaz de réseau + coke de pétrole + butane propane + fioul lourd + fioul domestique + gazole non routier. | 1 | Total combustibles usuels |
| INDIC_T1 | Consommation d'énergie en ktep | MR_CSTP |  | 1 | Autres produits pétroliers |
| INDIC_T1 | Consommation d'énergie en ktep | NR_CSTP |  | 1 | Liqueurs noires |
| INDIC_T1 | Consommation d'énergie en ktep | OR_CSTP |  | 1 | Bois et sous-produit du bois |
| INDIC_T1 | Consommation d'énergie en ktep | PR_CSTP |  | 1 | Hydrogene |
| INDIC_T1 | Consommation d'énergie en ktep | XR_CSTP |  | 1 | Combustibles spéciaux renouvelables |
| INDIC_T1 | Consommation d'énergie en ktep | YR_CSTP |  | 1 | Combustibles spéciaux non renouvelables |
| INDIC_T1 | Consommation d'énergie en ktep | CAR_CSTP | Total autres combustibles = autres produits pétroliers + liqueur noire + bois et sous-produits du bois + hydrogène + combustibles spéciaux renouvelables + combustibles spéciaux non renouvelables. | 1 | Total autres combustibles |
| INDIC_T1 | Consommation d'énergie en ktep | CR_CSTP |  | 1 | Achats de vapeur |
| INDIC_T1 | Consommation d'énergie en ktep | BR_CSTP |  | 1 | Electricité |
| INDIC_T1 | Consommation d'énergie en ktep | CONSBRT | Total brut = total combustibles + achats de vapeur + électricité | 1 | Total brut |
| INDIC_T1 | Consommation d'énergie en ktep | CR_VTTP |  | 1 | Vapeur vendue |
| INDIC_T1 | Consommation d'énergie en ktep | COMBPRB |  | 1 | Consommation pour production d'électricité |
| INDIC_T1 | Consommation d'énergie en ktep | CONSNET | Total net = Total brut - Vapeur vendue - Consommation pour production d'électricité. | 1 | Total net |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | FACTURE | en millions d’euros hors taxes | 1 | Valeur des achats de l’ensemble des énergies (en millions d'euros) |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_QAUP | en milliers de tonnes | 1 | Quantités achetées de combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_CSUP | en milliers de tonnes | 1 | Quantités consommées de combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CMSR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen des combustibles minéraux solides |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_QAUP | en GWh | 1 | Quantités achetées de gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_CSUP | en GWh | 1 | Quantités consommées de gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | GAZR_PRIX | en euros par MWh, hors taxes | 1 | Prix moyen du gaz |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_QAUP | en GWh | 1 | Quantités achetées de biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_CSUP | en GWh | 1 | Quantités consommées de biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BIOR_PRIX | en euros par MWh, hors taxes | 1 | Prix moyen du biogaz et biométhane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_QAUP | en milliers de tonnes | 1 | Quantités achetées de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_CSUP | en milliers de tonnes | 1 | Quantités consommées de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | IR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de coke de pétrole |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_QAUP | en milliers de tonnes | 1 | Quantités achetées de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_CSUP | en milliers de tonnes | 1 | Quantités consommées de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | JR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de butane-propane |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_QAUP | en milliers de tonnes | 1 | Quantités achetées de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_CSUP | en milliers de tonnes | 1 | Quantités consommées de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | KR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de fioul lourd |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_QAUP | en milliers de litres | 1 | Quantités achetées de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_CSUP | en milliers de litres | 1 | Quantités consommées de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | LR_PRIX | en euros par millier de litres, hors taxes | 1 | Prix moyen de fioul domestique |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_QAUP | en milliers de litres | 1 | Quantités achetées de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_CSUP | en milliers de litres | 1 | Quantités consommées de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | QR_PRIX | en euros par millier de litres, hors taxes | 1 | Prix moyen de gazole non routier |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_QAUP | en milliers de tonnes | 1 | Quantités achetées de bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_CSUP | en milliers de tonnes | 1 | Quantités consommées de bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | OR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen du bois |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CR_QAUP | en milliers de tonnes | 1 | Quantités achetées de vapeur |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CR_VALA | en millions d’euros, hors taxes | 1 | Valeur des achats de vapeur |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | CR_PRIX | en euros par tonne, hors taxes | 1 | Prix moyen de vapeur |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_QAUP | en milliers de tonnes | 1 | Quantités achetées d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_CSUP | en milliers de tonnes | 1 | Quantités consommées d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_PROD | en milliers de tonnes | 1 | Quantités autoproduites d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_PRIX | en euros par tonnes, hors taxe | 1 | Prix moyen d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_AUCO | en milliers de tonnes | 1 | Autoproduction consommée |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | PR_VEND | en milliers de tonnes | 1 | Autoproduction vendue d’hydrogène |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_QAUP | en GWh | 1 | Quantités achetées d'électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_CSUP | en GWh | 1 | Quantités consommées d'électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_PROD | en GWh | 1 | Quantités autoproduites d’électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_VALA | en millions d’euros hors taxes | 1 | Valeur des achats d'électricité |
| INDIC_T2 | Achats et consommation en unité propre, valeur d'achat et prix moyen des produits énergétiques | BR_PRIX | en euros par MWh, hors taxes | 1 | Prix moyen d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQFAB |  | 1 | Quantités de gaz consommées pour la fabrication |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ2 |  | 1 | Quantités de gaz consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ3 |  | 1 | Quantités de gaz consommées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ4 |  | 1 | Quantités de gaz consommées pour le chauffage ou un autre usage |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_USQ9 |  | 1 | Quantités de gaz consommées pour la mobilité sur site |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | GAZR_CSTP |  | 1 | Quantités de gaz consommées |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQFAB |  | 1 | Quantités de fioul domestique consommées pour la fabrication |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQ2 |  | 1 | Quantités de fioul domestique consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQ3 |  | 1 | Quantités de fioul domestique consommées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_USQ4 |  | 1 | Quantités de fioul domestique consommées pour le chauffage ou un autre usage |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | LR_CSTP |  | 1 | Quantités de fioul domestique consommées |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQFAB |  | 1 | Quantités de gazole non routier consommées pour la fabrication |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ2 |  | 1 | Quantités de gazole non routier consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ3 |  | 1 | Quantités de gazole non routier consommées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ4 |  | 1 | Quantités de gazole non routier consommées pour le chauffage ou un autre usage |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_USQ9 |  | 1 | Quantités de gazole non routier consommées pour la mobilité sur site |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | QR_CSTP |  | 1 | Quantités de gazole non routier consommées |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ2 |  | 1 | Quantités d’hydrogène consommées en tant que matières premières |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ8 |  | 1 | Quantités d’hydrogène utilisées pour la combustion (hors production d’électricité) |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ3 |  | 1 | Quantités d’hydrogène utilisées pour la production d’électricité |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_USQ9 |  | 1 | Quantités d’hydrogène utilisées pour la mobilité sur site |
| INDIC_T3 | Répartition de la consommation de combustibles par usage en ktep | PR_CSTP |  | 1 | Quantités d’hydrogène consommées |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_THER |  | 1 | Autoproduction d'origine thermique |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_NTHE | Autoproduction d'origine non thermique =Autoproduction d'origine hydraulique, photovoltaique ou éolienne. | 1 | Autoproduction d'origine non thermique |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_PROD |  | 1 | Autoproduction électricité |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_VEND |  | 1 | Autoproduction vendue |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_AUCO |  | 1 | Autoproduction consommée (1) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_QAUP |  | 1 | Achats (2) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_CSUP |  | 1 | Consommation (1 + 2) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ1 |  | 1 | Usage force motrice |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ2 |  | 1 | Usages thermiques (y compris production de froid) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ4 |  | 1 | Usage électrolyse |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ6 |  | 1 | Usage chauffage des locaux et autres |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ7 |  | 1 | Usages thermodynamiques hors chauffage des locaux (Prod. froid, PAC, CMV) |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ8 |  | 1 | Usage éclairage |
| INDIC_T4 | Autoproduction, achats et consommation d'électricité par usage en GWh | BR_USQ9 |  | 1 | Usage mobilité sur site |


## IV. Phase 1 — le modele analyse et pose ses questions

`chat()` envoie le prompt systeme + les metadonnees. Le modele repond soit directement
en JSON (aucune question), soit par une liste de questions.


In [5]:
prompt = open(PROMPT_PATH, encoding="utf-8").read()
history = [
    {"role": "system", "content": prompt},
    {"role": "user", "content": wrap(markdown_input)},
]

print("Phase 1 -- envoi au modele...\n")
reply = chat(history)
history.append({"role": "assistant", "content": reply})

auto_continued = is_auto_continued(reply)

if auto_continued:
    print("Le modele n'a pose aucune question -- JSON produit directement.")
    print("Passez directement a la cellule 'Verification' (la cellule 'Reponses' sera ignoree).")
else:
    print("-" * 70)
    print(reply)
    print("-" * 70)
    print("\n-> Remplissez la variable `answers` dans la cellule suivante, puis executez-la.")


Phase 1 -- envoi au modele...

----------------------------------------------------------------------
**Liste des feuilles et de leur rôle :**

1.  **Tableaux** : Feuille de demande. Liste les 12 tableaux demandés (`naf_T1` à `teff_T4`) avec leurs indicateurs et dimensions de croisement.
2.  **Variables** : Feuille de référence / nomenclature. Définit les codes et libellés pour les variables de croisement `REGION`, `SECTEUR` et `TEFF`.
3.  **Indicateurs** : Feuille de référence / nomenclature. Définit les codes et libellés pour les variables indicatrices `INDIC_T1`, `INDIC_T2`, `INDIC_T3` et `INDIC_T4`.

---

1. Champ et population
Pour les tableaux `naf_T1` à `teff_T4`, le champ est décrit comme « France, établissements de 20 salariés ou plus appartenant à l'industrie, hors industrie de l'énergie et hors artisanat commercial, y compris récupération ». Le terme « industrie » est-il à interpréter comme l'ensemble des divisions NAF listées dans la variable `SECTEUR` (07 à 38), ou corresp

## Reponses du producteur

Ne remplir que si des questions ont ete posees ci-dessus (une reponse par question, dans l'ordre).


In [7]:
answers = """
1) le champ est: 2023 
2) Ces agregats sont des indicateurs indépendants. ce qui ne doit pas paraitre c'est INDIC_T1 à INDIC_T4 qaui sont des groups d'indicateurs, PAS LES INDICATEURS EUX MEMES.
3) utilise simplement code = secteur et hrc_spanning = hrc_secteur; de meme respesctivement pour les autres variables de croisement 
4) renomme les par T1, T2,... T300 (il y a autour des 300 indicateur, pas exactement)

"""

print("Reponses enregistrees -- executez la cellule 'Verification' pour envoyer au modele.")


Reponses enregistrees -- executez la cellule 'Verification' pour envoyer au modele.


## V-VI. Verification et ecriture du CSV

Envoie les reponses si necessaire, valide le JSON contre le schema, puis ecrit directement le CSV.
Une seule cellule a executer une fois que le modele a fini de repondre.


In [8]:
if not auto_continued:
    print("Phase 2 -- envoi des reponses au modele...\n")
    history.append({"role": "user", "content": answers})
    reply = chat(history)

# ---- V. Verify -----------------------------------------------------
records = extract_array(reply)
if records is None:
    raise ValueError("Aucun tableau JSON trouve dans la reponse du modele.")

errors = validate(records)
if errors:
    raise ValueError("Schema validation failed:\n" + "\n".join(errors))

print(f"{len(records)} enregistrement(s) valides contre le schema.")

# ---- VI. Write CSV ---------------------------------------------------
import csv
import pandas as pd

def _open_any(path, mode="r", **kw):
    """open() local ou s3fs selon le prefixe du chemin."""
    if path.startswith("s3://"):
        from src.data import connect_s3
        return connect_s3().open(path, mode, **kw)
    return open(path, mode, **kw)

n_span = max_spanning(records)
cols = list(HEADER_BASE)
for i in range(1, n_span + 1):
    cols += [f"spanning_{i}", f"hrc_spanning_{i}"]

rows = []
for rec in records:
    row = [rec["table_name"], rec["field"], rec["hrc_field"],
           rec["indicator"], rec["hrc_indicator"]]
    pairs = _spanning_pairs(rec)
    for i in range(n_span):
        code_, hrc = pairs[i] if i < len(pairs) else ("", "")
        row += [code_, hrc]
    rows.append(row)

with _open_any(OUTPUT_PATH, "w", newline="", encoding="utf-8-sig") as fh:
    w = csv.writer(fh)
    w.writerow(cols)
    w.writerows(rows)

print(f"Ecrit {OUTPUT_PATH} ({len(rows)} lignes x {len(cols)} colonnes)")

with _open_any(OUTPUT_PATH, "r", encoding="utf-8-sig") as f:
    df = pd.read_csv(f, keep_default_na=False)
display(df)


Phase 2 -- envoi des reponses au modele...

300 enregistrement(s) valides contre le schema.
Ecrit s3://jawadmallat/analyse_LLM_metadata/6juillet/metadonnees_ex5_sansSL2.csv (300 lignes x 9 colonnes)


,table_name,field,hrc_field,indicator,hrc_indicator,spanning_1,hrc_spanning_1,spanning_2,hrc_spanning_2
0,T1,etablissements_20_salaries_ou_plus_industrie_h...,NA,NB_ETAB,NA,secteur,hrc_secteur,INDIC_T1,NA
1,T2,etablissements_20_salaries_ou_plus_industrie_h...,NA,CMSR_CSTP,NA,secteur,hrc_secteur,INDIC_T1,NA
2,T3,etablissements_20_salaries_ou_plus_industrie_h...,NA,GAZR_CSTP,NA,secteur,hrc_secteur,INDIC_T1,NA
3,T4,etablissements_20_salaries_ou_plus_industrie_h...,NA,BIOR_CSTP,NA,secteur,hrc_secteur,INDIC_T1,NA
4,T5,etablissements_20_salaries_ou_plus_industrie_h...,NA,IR_CSTP,NA,secteur,hrc_secteur,INDIC_T1,NA
...,...,...,...,...,...,...,...,...,...
295,T296,etablissements_20_salaries_ou_plus_industrie_h...,NA,BR_VALA,NA,TEFF,hrc_teff,INDIC_T2,NA
296,T297,etablissements_20_salaries_ou_plus_industrie_h...,NA,BR_PRIX,NA,TEFF,hrc_teff,INDIC_T2,NA
297,T298,etablissements_20_salaries_ou_plus_industrie_h...,NA,GAZR_USQFAB,NA,TEFF,hrc_teff,INDIC_T3,NA
298,T299,etablissements_20_salaries_ou_plus_industrie_h...,NA,GAZR_USQ2,NA,TEFF,hrc_teff,INDIC_T3,NA


## Evaluation de qualite

Compare la sortie du pipeline avec `REFERENCE_CSV` (sortie attendue, corrigee a la main) et
compte les cellules incorrectes, colonne par colonne. Chaque execution ajoute une ligne dans
`notebook/results.csv` pour suivre si le modele se trompe systematiquement sur une variable
donnee plutot qu'aleatoirement.

Necessite `REFERENCE_CSV` renseigne dans la cellule de configuration ci-dessus.


In [9]:
import re
from datetime import datetime, timezone

if REFERENCE_CSV is None:
    print("REFERENCE_CSV n'est pas renseigne -- etape sautee.")
else:
    def _open_any(path, mode="r", **kw):
        if path.startswith("s3://"):
            from src.data import connect_s3
            return connect_s3().open(path, mode, **kw)
        return open(path, mode, **kw)

    RESULTS_CSV = REPO_ROOT / "notebook" / "results.csv"
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)

    with _open_any(REFERENCE_CSV, "r", encoding="utf-8-sig") as f:
        ref = pd.read_csv(f, keep_default_na=False)
    out = df.copy()

    # -- Comparaison du nombre de tableaux --------------------------------
    n_tables_ref = len(ref)
    n_tables_out = len(out)
    n_tables_diff = n_tables_out - n_tables_ref

    # -- Alignement sur table_name (repli positionnel) --------------------
    def _align(ref_df, out_df):
        if "table_name" in ref_df.columns and "table_name" in out_df.columns:
            ref_ids = ref_df["table_name"].tolist()
            out_lookup = set(out_df["table_name"])
            common = [t for t in ref_ids if t in out_lookup]
            if common:
                ra = ref_df.set_index("table_name").reindex(common)
                oa = out_df.set_index("table_name").reindex(common)
                return ra, oa, len(ref_df) - len(common), len(out_df) - len(common)
        n = min(len(ref_df), len(out_df))
        return (ref_df.iloc[:n].reset_index(drop=True),
                out_df.iloc[:n].reset_index(drop=True),
                len(ref_df) - n, len(out_df) - n)

    ref_a, out_a, n_miss_ref, n_miss_out = _align(ref, out)

    # Normalise les cellules vides en "NA" (sentinel du schema)
    ref_a = ref_a.fillna("NA").replace("", "NA")
    out_a = out_a.fillna("NA").replace("", "NA")

    # -- Comptage des cellules incorrectes par colonne ---------------------
    SPAN_RE = re.compile(r"^spanning_\d+$")
    HRC_SPAN_RE = re.compile(r"^hrc_spanning_\d+$")

    col_errors = {"table_name": 0, "field": 0, "hrc_field": 0,
                  "indicator": 0, "hrc_indicator": 0,
                  "spanning": 0, "hrc_spanning": 0}

    for col in ref_a.columns:
        if col not in out_a.columns:
            continue
        n_wrong = int((ref_a[col].astype(str) != out_a[col].astype(str)).sum())
        if col in col_errors:
            col_errors[col] += n_wrong
        elif SPAN_RE.match(col):
            col_errors["spanning"] += n_wrong
        elif HRC_SPAN_RE.match(col):
            col_errors["hrc_spanning"] += n_wrong

    unmatched_err = (n_miss_ref + n_miss_out) * max(len(ref_a.columns), 1)
    total_wrong = sum(col_errors.values()) + unmatched_err

    # -- Resume -------------------------------------------------------------
    diff_label = "=" if n_tables_diff == 0 else f"{n_tables_diff:+d}"
    print(f"  Tableaux attendus  : {n_tables_ref}")
    print(f"  Tableaux produits  : {n_tables_out}  [{diff_label}]")
    if n_tables_diff != 0:
        print(f"  !! LLM a {'ajoute' if n_tables_diff > 0 else 'omis'} {abs(n_tables_diff)} tableau(x)")
    print(f"\n  {'Colonne':<22} {'Cellules incorrectes':>20}")
    print("  " + "-" * 44)
    for k, v in col_errors.items():
        print(f"  {k:<22} {v:>20}")
    if n_miss_ref or n_miss_out:
        print(f"\n  Lignes manquantes : {n_miss_ref}")
        print(f"  Lignes en exces   : {n_miss_out}")
    print(f"\n  -> TOTAL : {total_wrong} cellules incorrectes")

    # -- results.csv ----------------------------------------------------------
    RESULT_COLS = [
        "filename", "timestamp", "total_wrong_cells",
        "n_tables_ref", "n_tables_out", "n_tables_diff",
        "table_name", "field", "hrc_field",
        "indicator", "hrc_indicator",
        "spanning", "hrc_spanning",
        "missing_from_output", "extra_in_output",
    ]
    row = {
        "filename": Path(INPUT_PATH).name,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "total_wrong_cells": total_wrong,
        "n_tables_ref": n_tables_ref,
        "n_tables_out": n_tables_out,
        "n_tables_diff": n_tables_diff,
        **col_errors,
        "missing_from_output": n_miss_ref,
        "extra_in_output": n_miss_out,
    }

    write_header = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=RESULT_COLS, extrasaction="ignore")
        if write_header:
            w.writeheader()
        w.writerow(row)


  Tableaux attendus  : 312
  Tableaux produits  : 300  [-12]
  !! LLM a omis 12 tableau(x)

  Colonne                Cellules incorrectes
  --------------------------------------------
  table_name                              300
  field                                   300
  hrc_field                                 0
  indicator                               296
  hrc_indicator                             0
  spanning                                300
  hrc_spanning                            300

  Lignes manquantes : 12
  Lignes en exces   : 0

  -> TOTAL : 1556 cellules incorrectes
